# The Blockbuster Formula
## Causal Analysis of Box Office Success Using Bayesian Networks

**Author:** Steve Meta  
**Program:** Master's in Data Science â€” IE University  
**Date:** April 2026

---

### Research Question
> *Do high-profile casts and large production/marketing budgets causally drive box office success, or are other factors (genre, release timing, audience reception) stronger drivers?*

### Approach
Rather than treating this as a pure prediction problem, we use a **Bayesian Network** to model the *causal structure* of box office success. This allows us to:
- Understand *why* certain movies succeed, not just predict which ones will
- Run counterfactual queries ("what if we swapped the cast?")
- Quantify the relative influence of each factor

### Notebook Sections
1. Data Collection & Loading
2. Data Cleaning & Preprocessing
3. Feature Engineering
4. Exploratory Data Analysis (EDA)
5. Bayesian Network (DAG + Fitting)
6. Probabilistic Inference & Counterfactual Reasoning
7. Baseline Model Comparison (Logistic Regression & Random Forest)
8. Sensitivity Analysis & Conclusion

---
## Section 1 â€” Data Collection & Loading

### What we're doing
We pull movie data from **The Movie Database (TMDb) API** â€” the most comprehensive freely available source for film metadata. For each movie released between 2000 and 2024, we collect:

- **Identity**: title, TMDb ID, release date
- **Financials**: production budget, worldwide revenue
- **Creative**: genres, top-billed cast (up to 5 actors)
- **Reception**: TMDb popularity score, vote average, vote count

### Why TMDb?
TMDb provides structured, machine-readable data via a well-documented REST API with a generous free tier. Budget/revenue data comes from the movie's `/details` endpoint, while cast data comes from the `/credits` endpoint. We page through `discover/movie` sorted by popularity to capture blockbuster-tier films.

### Data volume
We target **~3,000â€“5,000 movies** across 25 years. After filtering for budget > $10M in Section 2, we expect ~1,500â€“2,500 true blockbusters.

In [12]:
# â”€â”€ Imports â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import os
import time
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from tqdm.notebook import tqdm

# â”€â”€ Load API key from .env â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
load_dotenv()
TMDB_API_KEY = os.getenv('TMDB_API_KEY')

if not TMDB_API_KEY:
    raise ValueError("TMDb API key not found. Make sure .env contains TMDB_API_KEY=your_key")

#just making sure it's well loaded
#print(f"API key loaded: {TMDB_API_KEY[:6]}{'*' * (len(TMDB_API_KEY) - 6)}")

# â”€â”€ Directory setup â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
Path('data').mkdir(exist_ok=True)
Path('outputs').mkdir(exist_ok=True)
print("Directories ready: data/ and outputs/")

Directories ready: data/ and outputs/


In [ ]:
# ── Smart loader: skip API calls if CSV already exists ───────────────────────
# On re-opening the notebook, run THIS cell first.
# If data/movies_raw.csv exists, it loads directly and skips all API calls.

RAW_PATH = Path("data/movies_raw.csv")

if RAW_PATH.exists():
    df_raw = pd.read_csv(RAW_PATH, parse_dates=["release_date"])
    print(f"Loaded existing data from {RAW_PATH}")
    print(f"Shape: {df_raw.shape}  |  Date range: {df_raw["release_date"].min().year} – {df_raw["release_date"].max().year}")
    print("Skipping API collection — jump straight to Section 2.")
else:
    print(f"{RAW_PATH} not found — run the API collection cells below to build it.")
    df_raw = None

In [13]:
# â”€â”€ TMDb API configuration â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
BASE_URL   = 'https://api.themoviedb.org/3'
HEADERS    = {'accept': 'application/json'}

# Years to collect (inclusive)
START_YEAR = 2000
END_YEAR   = 2024

# Pages per year â€” TMDb returns 20 movies/page, sorted by popularity desc.
# 10 pages Ã— 20 = 200 movies per year Ã— 25 years = up to 5,000 movies
PAGES_PER_YEAR = 10

# Politeness delay between API calls (seconds)
REQUEST_DELAY = 0.25

def tmdb_get(endpoint: str, params: dict = None) -> dict | None:
    """Make a GET request to the TMDb API. Returns JSON dict or None on error.
    Uses v3 API key authentication â€” passed as a query parameter, not Bearer token.
    """
    url = f"{BASE_URL}/{endpoint}"
    # v3 keys go in as ?api_key=xxx, not as Authorization: Bearer
    all_params = {'api_key': TMDB_API_KEY}
    if params:
        all_params.update(params)
    try:
        response = requests.get(url, headers=HEADERS, params=all_params, timeout=10)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Request failed for {endpoint}: {e}")
        return None

# â”€â”€ Quick connectivity test â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
test = tmdb_get('configuration')
if test:
    print("TMDb API connection successful!")
else:
    print("Connection failed â€” check your API key.")

TMDb API connection successful!


In [14]:
# â”€â”€ Step 1: Discover movie IDs by year â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# We use /discover/movie sorted by popularity descending, filtered by year.
# This gives us the most-discussed films for each year â€” exactly what we want
# for a blockbuster analysis.

def discover_movie_ids(year: int, pages: int = PAGES_PER_YEAR) -> list[int]:
    """Return a list of TMDb movie IDs for a given release year."""
    ids = []
    for page in range(1, pages + 1):
        data = tmdb_get('discover/movie', params={
            'primary_release_year': year,
            'sort_by': 'popularity.desc',
            'include_adult': False,
            'include_video': False,
            'page': page,
            'language': 'en-US',
            'with_original_language': 'en'  # English-language films only
        })
        if data and 'results' in data:
            ids.extend([m['id'] for m in data['results']])
        time.sleep(REQUEST_DELAY)
    return ids

# Collect IDs across all years
print(f"Discovering movie IDs from {START_YEAR} to {END_YEAR}...")
all_ids = []

for year in tqdm(range(START_YEAR, END_YEAR + 1), desc="Years"):
    year_ids = discover_movie_ids(year)
    all_ids.extend(year_ids)

# Remove duplicates (some films appear across year boundaries)
all_ids = list(set(all_ids))
print(f"\nTotal unique movie IDs collected: {len(all_ids)}")

Discovering movie IDs from 2000 to 2024...


Years:   0%|          | 0/25 [00:00<?, ?it/s]


Total unique movie IDs collected: 4996


In [15]:
# â”€â”€ Step 2: Fetch detailed data for each movie â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# For each ID we make two calls:
#   1. /movie/{id}           â†’ budget, revenue, genres, runtime, tagline
#   2. /movie/{id}/credits   â†’ cast (top 5 billed actors)

def fetch_movie_details(movie_id: int) -> dict | None:
    """Fetch full details + credits for a single TMDb movie ID."""
    details = tmdb_get(f'movie/{movie_id}', params={'language': 'en-US'})
    if not details:
        return None

    # Fetch cast credits
    credits = tmdb_get(f'movie/{movie_id}/credits', params={'language': 'en-US'})
    cast_list = []
    cast_ids  = []
    if credits and 'cast' in credits:
        # Top 5 billed actors
        top_cast = sorted(credits['cast'], key=lambda x: x.get('order', 99))[:5]
        cast_list = [c['name'] for c in top_cast]
        cast_ids  = [c['id'] for c in top_cast]

    # Extract genre names
    genres = [g['name'] for g in details.get('genres', [])]

    return {
        'tmdb_id':           movie_id,
        'title':             details.get('title', ''),
        'release_date':      details.get('release_date', ''),
        'genres':            '|'.join(genres),
        'budget':            details.get('budget', 0),
        'revenue':           details.get('revenue', 0),
        'runtime':           details.get('runtime', 0),
        'popularity':        details.get('popularity', 0),
        'vote_average':      details.get('vote_average', 0),
        'vote_count':        details.get('vote_count', 0),
        'original_language': details.get('original_language', ''),
        'tagline':           details.get('tagline', ''),
        'overview':          details.get('overview', ''),
        'cast_names':        '|'.join(cast_list),
        'cast_ids':          '|'.join(str(i) for i in cast_ids)
    }

# â”€â”€ Collect all movie records â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print(f"Fetching details for {len(all_ids)} movies...")
print("This will take approximately", round(len(all_ids) * REQUEST_DELAY * 2 / 60, 1), "minutes.")

movies = []
failed = []

for movie_id in tqdm(all_ids, desc="Movies"):
    record = fetch_movie_details(movie_id)
    if record:
        movies.append(record)
    else:
        failed.append(movie_id)
    time.sleep(REQUEST_DELAY)

print(f"\nSuccessfully fetched: {len(movies)} movies")
print(f"Failed:              {len(failed)} movies")

Fetching details for 4996 movies...
This will take approximately 41.6 minutes.


Movies:   0%|          | 0/4996 [00:00<?, ?it/s]


Successfully fetched: 4996 movies
Failed:              0 movies


In [20]:
# ── Step 3: Build DataFrame, save immediately, and preview ───────────────────
df_raw = pd.DataFrame(movies)

# Convert release_date to datetime
df_raw["release_date"] = pd.to_datetime(df_raw["release_date"], errors="coerce")

# Add release year as a convenience column
df_raw["release_year"] = df_raw["release_date"].dt.year

# Save immediately so we never have to re-run the API calls again
RAW_PATH = Path("data/movies_raw.csv")
df_raw.to_csv(RAW_PATH, index=False)
print(f"Saved {len(df_raw)} rows to {RAW_PATH}")

# Preview
print(f"Shape: {df_raw.shape}  |  Columns: {len(df_raw.columns)}")
df_raw.head(3)

Saved 4996 rows to data\movies_raw.csv
Shape: (4996, 16)  |  Columns: 16


,tmdb_id,title,release_date,genres,budget,revenue,runtime,popularity,vote_average,vote_count,original_language,tagline,overview,cast_names,cast_ids,release_year
0,12,Finding Nemo,2003-05-30,Animation|Family|Adventure,94000000,940335536,100,17.0226,7.817,20421,en,There are 3.7 trillion fish in the ocean. They...,"Nemo, an adventurous young clownfish, is unexp...",Albert Brooks|Ellen DeGeneres|Alexander Gould|...,13|14|12|5293|118,2003
1,16,Dancer in the Dark,2000-09-01,Drama|Crime,12500000,40061153,140,3.0415,7.845,1981,en,"In a world of shadows, she found the light of ...","Selma, a Czech immigrant on the verge of blind...",Björk|Catherine Deneuve|David Morse|Peter Stor...,47|50|52|53|6748,2000
2,22,Pirates of the Caribbean: The Curse of the Bla...,2003-07-09,Adventure|Fantasy|Action,140000000,655011224,143,16.3728,7.821,22054,en,Prepare to be blown out of the water.,When wily pirate Captain Barbossa seizes Jack ...,Johnny Depp|Geoffrey Rush|Orlando Bloom|Keira ...,85|118|114|116|1709,2003


In [21]:
# Basic stats on raw data
print("=== Raw Data Summary ===")
print(f"Total movies:          {len(df_raw)}")
print(f"Movies with budget>0:  {(df_raw["budget"] > 0).sum()}")
print(f"Movies with revenue>0: {(df_raw["revenue"] > 0).sum()}")
print(f"Date range:            {df_raw["release_date"].min().date()} to {df_raw["release_date"].max().date()}")
print()

# Budget/revenue distribution (non-zero only)
budget_nz = df_raw[df_raw["budget"] > 0]["budget"]
print(f"Budget  - min: ${budget_nz.min():>15,.0f}  |  median: ${budget_nz.median():>12,.0f}  |  max: ${budget_nz.max():>15,.0f}")

revenue_nz = df_raw[df_raw["revenue"] > 0]["revenue"]
print(f"Revenue - min: ${revenue_nz.min():>15,.0f}  |  median: ${revenue_nz.median():>12,.0f}  |  max: ${revenue_nz.max():>15,.0f}")

=== Raw Data Summary ===
Total movies:          4996
Movies with budget>0:  3740
Movies with revenue>0: 3794
Date range:            2000-01-01 to 2024-12-25

Budget  - min: $              5  |  median: $  30,000,000  |  max: $    489,900,000
Revenue - min: $              7  |  median: $  52,729,797  |  max: $  2,923,706,026
